# DocsMind: inspect the built RAG pipeline

This notebook is a visual tour of the code that is already in the repository.

It follows the real query path:

**Load → Chunk → Embed → Index → Search → BM25 → Fuse → Rerank → Context → Generate → Cite**

The notebook uses the existing project functions. The default LLM cell uses a tiny inspection stub so it stays offline and does not spend API credits. An optional final cell calls whichever real LLM provider is active in the repository configuration.

The notebook reads the active corpus and chunk settings from the repository configuration, so it follows the same path as the production ingest script.


## How to use this notebook

Install the notebook environment once from the repository root:

```bash
.venv/bin/pip install -e ".[dev,notebook]"
make notebook
```

Then run the cells from top to bottom.

The most useful things to inspect are:

- how document metadata becomes chunk provenance;
- the shape and normalization of embedding vectors;
- how dense search and BM25 return different rankings;
- how RRF creates one hybrid ranking;
- how the pipeline turns retrieved chunks into a numbered prompt;
- how citation markers map back to source chunks.

The cross-encoder reranker is present but off by default because it downloads a model. Set RUN_RERANK = True only when you want to inspect that stage.


In [ ]:
from pathlib import Path
import html
import json
import logging
import os
import re
import sys
import warnings

from IPython.display import HTML, Markdown, display

# Keep notebook output focused on the pipeline rather than model-loader logs.
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
logging.getLogger().setLevel(logging.WARNING)
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("sentence_transformers").setLevel(logging.WARNING)
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)
logging.getLogger("faiss.loader").setLevel(logging.WARNING)
warnings.filterwarnings("ignore", message="IProgress not found.*")
warnings.filterwarnings("ignore", message="Warning: You are sending unauthenticated requests.*")
warnings.filterwarnings("ignore", category=FutureWarning, module="docsmind.index.embeddings")

# Find the repository root even if Jupyter was started from a subdirectory.
ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "docsmind").exists():
    ROOT = ROOT.parent

if not (ROOT / "docsmind").exists():
    raise RuntimeError("Start Jupyter from the DocsMind repository, or set ROOT manually.")

sys.path.insert(0, str(ROOT))

from docsmind.config import Settings

repo_settings = Settings(_env_file=ROOT / ".env")
DATA_DIR = repo_settings.data_dir
if not DATA_DIR.is_absolute():
    DATA_DIR = ROOT / DATA_DIR
print(f"Repository: {ROOT}")
print(f"Corpus: {DATA_DIR}")


In [ ]:
def show_table(rows, columns=None):
    """Small notebook-only table helper; avoids adding a pandas dependency."""
    if not rows:
        display(Markdown("_No rows to display._"))
        return
    columns = columns or list(rows[0].keys())
    header = "".join(f"<th>{html.escape(str(c))}</th>" for c in columns)
    body = ""
    for row in rows:
        body += "<tr>" + "".join(
            f"<td>{html.escape(str(row.get(c, '')))}</td>" for c in columns
        ) + "</tr>"
    display(HTML(
        "<table style='border-collapse:collapse'>"
        f"<thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>"
    ))

def show_results(results, limit=5):
    rows = []
    for rank, result in enumerate(results[:limit], start=1):
        rows.append({
            "rank": rank,
            "score": round(result.score, 4),
            "source": result.chunk.source,
            "chunk_id": result.chunk.id[:12],
            "preview": result.chunk.text[:180].replace("\n", " "),
        })
    show_table(rows)


## 1. Ingest: load the source documents

This is the Ingest stage. LlamaIndex reads the files and attaches metadata such as the filename. That metadata is what later makes citations possible.


In [ ]:
from docsmind.ingestion.loaders import load_documents

documents = load_documents(
    DATA_DIR,
    whatsapp_window_minutes=repo_settings.whatsapp_window_minutes,
    whatsapp_max_messages=repo_settings.whatsapp_max_messages,
)

print(f"Loaded {len(documents)} documents")
show_table(
    [
        {
            "source": doc.metadata.get("file_name") or doc.metadata.get("file_path", "unknown"),
            "characters": len(doc.text),
            "metadata_keys": ", ".join(sorted(doc.metadata.keys())),
        }
        for doc in documents
    ],
    columns=["source", "characters", "metadata_keys"],
)


In [ ]:
# See one complete loaded document before chunking.
example_doc = documents[0]
display(Markdown(
    f"### {example_doc.metadata.get('file_name', 'document')}\n\n"
    + example_doc.text[:2000]
    + ("\n\n..." if len(example_doc.text) > 2000 else "")
))


## 2. Chunk: split documents into retrieval-sized pieces

This is the Chunk stage. The notebook uses the active chunk size and overlap from the repository configuration.

The overlap keeps context from being cut too sharply at boundaries.


In [ ]:
from docsmind.ingestion.chunker import chunk_documents

CHUNK_SIZE = repo_settings.chunk_size
CHUNK_OVERLAP = repo_settings.chunk_overlap

chunks = chunk_documents(
    documents,
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)

print(f"Produced {len(chunks)} chunks")
show_table(
    [
        {
            "position": i,
            "source": chunk.source,
            "characters": len(chunk.text),
            "chunk_id": chunk.id[:12],
            "preview": chunk.text[:220].replace("\n", " "),
        }
        for i, chunk in enumerate(chunks[:12])
    ],
    columns=["position", "source", "characters", "chunk_id", "preview"],
)


In [ ]:
# A compact view of the chunk-size distribution.
lengths = [len(chunk.text) for chunk in chunks]
show_table([
    {
        "statistic": "minimum characters",
        "value": min(lengths),
    },
    {
        "statistic": "median characters",
        "value": sorted(lengths)[len(lengths) // 2],
    },
    {
        "statistic": "maximum characters",
        "value": max(lengths),
    },
])


## 3. Embed: turn each chunk into a normalized vector

This is the Embed stage. The same embedding model is used for documents and queries, so they can be compared in one vector space.

bge-small produces 384-dimensional vectors. The project normalizes them, which lets FAISS inner-product search behave like cosine similarity.


In [ ]:
from docsmind.index.embeddings import Embedder

embedder = Embedder(
    repo_settings.embed_model,
    device=repo_settings.embed_device or None,
)
embeddings = embedder.encode([chunk.text for chunk in chunks])

print(f"Embedding matrix shape: {embeddings.shape}")
print(f"Vector dtype: {embeddings.dtype}")
print(f"Model dimension reported by Embedder.dim: {embedder.dim}")

norms = (embeddings ** 2).sum(axis=1) ** 0.5
show_table([
    {"check": "first vector L2 norm", "value": round(float(norms[0]), 6)},
    {"check": "minimum norm", "value": round(float(norms.min()), 6)},
    {"check": "maximum norm", "value": round(float(norms.max()), 6)},
])


## 4. Index: build the FAISS vector store

This is the Index stage. FaissVectorStore stores both the vectors and the original Chunk objects.

The default flat index is exact: it compares the query against every vector. That is appropriate for this small corpus and gives us a clean baseline.


In [ ]:
from docsmind.index.faiss_store import FaissVectorStore

store = FaissVectorStore(dim=embedder.dim, index_type="flat")
store.add(chunks, embeddings)

print(f"Index type: {store.index_type}")
print(f"Indexed vectors: {store.size}")
print(f"Stored chunks: {len(store.chunks)}")


## 5. Search: dense retrieval

At query time, the question is embedded with the same model. FAISS then returns the chunks with the highest inner-product score.

This is the first ranking we will compare with BM25.


In [ ]:
question = "What symptoms were mentioned when someone asked if it was clutch slip?"
CANDIDATE_K = 20  # Wide retrieval before fusion
TOP_K = 5         # Final chunks we want to inspect

query_vector = embedder.encode([question])[0]
dense_results = store.search(query_vector, top_k=CANDIDATE_K)

print(f"Question: {question}")
show_results(dense_results, limit=TOP_K)


In [ ]:
# Inspect the score range and the actual text of the best dense match.
best_dense = dense_results[0]
print("Best dense match")
print(f"Score: {best_dense.score:.4f}")
print(f"Source: {best_dense.chunk.source}")
print()
print(best_dense.chunk.text)


## 6. Search beside dense retrieval: BM25

BM25 is lexical retrieval. It looks for matching terms and gives more weight to terms that are useful for distinguishing documents.

Dense retrieval and BM25 fail differently:

- dense retrieval handles paraphrases;
- BM25 handles exact names, terms, versions, and rare words.

BM25Index is rebuilt from the chunks at startup.


In [ ]:
from docsmind.retrieval.bm25 import BM25Index

bm25 = BM25Index(store.chunks)
bm25_results = bm25.search(question, top_k=CANDIDATE_K)

show_results(bm25_results, limit=TOP_K)


## 7. Fuse the rankings with Reciprocal Rank Fusion

Dense and BM25 scores are not on the same scale, so the system does not add their raw scores.

RRF uses rank positions instead:

1 / (fusion_k + zero-based rank)

A chunk that appears near the top in both lists receives support from both retrievers.


In [ ]:
from docsmind.retrieval.fusion import reciprocal_rank_fusion

hybrid_results = reciprocal_rank_fusion(
    [dense_results, bm25_results],
    k=60,
)

show_results(hybrid_results, limit=TOP_K)


In [ ]:
# Compare the three rankings side by side by chunk ID.
def rank_map(results):
    return {result.chunk.id: rank for rank, result in enumerate(results, start=1)}

dense_ranks = rank_map(dense_results)
bm25_ranks = rank_map(bm25_results)
hybrid_ranks = rank_map(hybrid_results)

all_ids = list(dict.fromkeys(
    [result.chunk.id for result in dense_results]
    + [result.chunk.id for result in bm25_results]
    + [result.chunk.id for result in hybrid_results]
))

comparison = []
for chunk_id in all_ids:
    chunk = next(chunk for chunk in store.chunks if chunk.id == chunk_id)
    comparison.append({
        "source": chunk.source,
        "chunk_id": chunk_id[:12],
        "dense_rank": dense_ranks.get(chunk_id, "—"),
        "bm25_rank": bm25_ranks.get(chunk_id, "—"),
        "hybrid_rank": hybrid_ranks.get(chunk_id, "—"),
    })

show_table(comparison)


## 8. Optional reranking

The repository also has a cross-encoder reranker. It runs after fusion and scores each question/chunk pair together.

That is more expensive than embedding search, so it is applied only to a small candidate set. The following cell is deliberately disabled by default because the model download can be large.


In [ ]:
RUN_RERANK = False

if RUN_RERANK:
    from docsmind.retrieval.reranker import CrossEncoderReranker

    reranker = CrossEncoderReranker(
        "cross-encoder/ms-marco-MiniLM-L-6-v2"
    )
    reranked_results = reranker.rerank(question, hybrid_results, top_k=5)
    show_results(reranked_results)
else:
    print("Reranking is OFF. Set RUN_RERANK = True to download and run the cross-encoder.")


## 9. Run the repository's HybridRetriever

The previous cells called the lower-level functions individually. This cell calls the production orchestration object that factory.py wires into the pipeline.

It should produce the same hybrid ordering as the manual dense → BM25 → RRF cells above.


In [ ]:
from docsmind.retrieval.retriever import HybridRetriever

retriever = HybridRetriever(
    embedder,
    store,
    candidate_k=20,
    fusion_k=60,
)

retriever_results = retriever.retrieve(question, top_k=TOP_K)
show_results(retriever_results)

print(
    "Matches manual fusion:",
    [r.chunk.id for r in retriever_results] == [r.chunk.id for r in hybrid_results[:5]],
)


## 10. Assemble the context sent to the LLM

This is the boundary between Retrieve and Generate.

RAGPipeline._build_context() numbers the selected chunks. The LLM later uses those numbers when it writes citations.


In [ ]:
from docsmind.pipeline import RAGPipeline, SYSTEM_PROMPT

context = RAGPipeline._build_context(retriever_results[:4])

display(Markdown("### System message"))
display(Markdown(SYSTEM_PROMPT))

display(Markdown("### User-side prompt context"))
display(Markdown(context))


In [ ]:
prompt = f"Context passages:\n\n{context}\n\nQuestion: {question}\n\nAnswer:"

display(HTML(
    "<pre style='white-space:pre-wrap; max-height:420px; overflow:auto'>"
    + html.escape(prompt)
    + "</pre>"
))


## 11. Exercise the full pipeline without calling a real LLM

This small stub implements the same LLMClient interface as Anthropic, Ollama, and vLLM. It lets us execute the real RAGPipeline.query() method and inspect its citation parsing without network access.

The production pipeline can swap this stub for the configured cloud, Ollama, or vLLM client.


In [ ]:
from docsmind.config import Settings
from docsmind.llm.base import LLMClient

class NotebookLLM(LLMClient):
    model = "notebook-inspection-stub"

    def __init__(self):
        self.last_system = None
        self.last_prompt = None
        self.last_max_tokens = None

    def generate(self, system: str, prompt: str, max_tokens: int) -> str:
        self.last_system = system
        self.last_prompt = prompt
        self.last_max_tokens = max_tokens
        # Deliberately cite the first numbered passage.
        return "The retrieved context contains a relevant discussion about the question [1]."

notebook_llm = NotebookLLM()
settings = Settings(top_k=4, max_tokens=128)

pipeline = RAGPipeline(retriever, notebook_llm, settings)
response = pipeline.query(question, top_k=4)

display(Markdown("### Pipeline answer"))
display(Markdown(response.answer))

show_table([
    {"field": "model", "value": response.model},
    {"field": "grounded", "value": response.grounded},
    {"field": "latency_ms", "value": round(response.latency_ms, 3)},
])


In [ ]:
display(Markdown("### Parsed citation objects"))
show_table([citation.model_dump() for citation in response.citations])

display(Markdown(
    "The stub saw this many prompt characters: "
    + str(len(notebook_llm.last_prompt))
))


## 12. Optional real configured LLM call

Set RUN_LIVE_LLM = True only when the configured model server or cloud provider is available. On the beast, the active provider is vLLM serving the model alias openclaw.

This is the only cell in the notebook that makes a model-server request.


In [ ]:
RUN_LIVE_LLM = False

if RUN_LIVE_LLM:
    from docsmind.factory import build_llm

    live_settings = repo_settings.model_copy(update={"top_k": 4, "max_tokens": 256})
    live_llm = build_llm(live_settings)
    live_pipeline = RAGPipeline(retriever, live_llm, live_settings)
    live_response = live_pipeline.query(question, top_k=4)

    display(Markdown(live_response.answer))
    show_table([citation.model_dump() for citation in live_response.citations])
else:
    print("Live LLM call is OFF. Set RUN_LIVE_LLM = True when the provider is ready.")


## What this walkthrough shows

The existing repository already has a complete retrieval path:

- documents retain source metadata;
- chunks retain that provenance;
- embeddings make semantic search possible;
- FAISS provides the dense index;
- BM25 adds exact-term search;
- RRF combines independent rankings;
- reranking is available as a gated final retrieval step;
- the pipeline numbers context passages;
- the LLM returns text containing citation markers;
- the pipeline maps those markers back to structured sources.

A useful next inspection is to change CHUNK_SIZE, question, and top_k, then observe how the ranking and context change. That is the fastest way to connect configuration dials to retrieval behavior.
